# AsanaAbhyas: yoga pose detection and classification

This notebook detects 33 body landmarks with the MediaPipe **Pose Landmarker** (Tasks API),
computes joint angles, and classifies three yoga poses with simple angle rules:
**Warrior II**, **T Pose** and **Tree Pose**.

> Ported from the legacy `mp.solutions.pose` API, which was removed from MediaPipe.
> The reusable code lives in the `asana_abhyas` package next to this notebook.

**Images:** the repository does not ship photos. Put your own `.jpg`/`.png` files in a `media/`
folder (or set the `ASANA_MEDIA_DIR` environment variable). Each photo should show one person,
full body, facing the camera.

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from asana_abhyas import PoseLandmark, calculate_angle, classify_pose
from asana_abhyas.detector import PoseDetector
from asana_abhyas.drawing import draw_label, draw_landmarks
from asana_abhyas.landmarks import POSE_CONNECTIONS

MEDIA_DIR = Path(os.environ.get("ASANA_MEDIA_DIR", "media"))
IMAGES = sorted(p for p in MEDIA_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"{len(IMAGES)} image(s) in {MEDIA_DIR.resolve()}")
if not IMAGES:
    print("Add some photos to the media/ folder to run the image cells below.")

## Initialize the pose detection model

`PoseDetector` wraps `mediapipe.tasks.python.vision.PoseLandmarker`. The model (`lite`, `full`
or `heavy`, roughly the old `model_complexity` 0/1/2) is downloaded once to
`~/.cache/asana_abhyas/`.

- `video=False` corresponds to the old `static_image_mode=True` and suits unrelated photos.
- `video=True` tracks the person between frames (webcam/video).
- `min_confidence` is used for detection, presence and tracking.

In [ ]:
detector = PoseDetector(model="heavy", video=False, min_confidence=0.3)

## Detect landmarks in one image

`detect()` returns 33 `(x_px, y_px, z)` tuples (pixel coordinates, with `z` on the same scale as
`x`) plus the world landmarks in metres, with the origin between the hips.

In [ ]:
if IMAGES:
    sample_img = cv2.imread(str(IMAGES[0]))
    landmarks, world = detector.detect(sample_img)
    if landmarks is None:
        print("No person detected")
    else:
        for lm in (PoseLandmark.NOSE, PoseLandmark.LEFT_EYE_INNER):
            x, y, z = landmarks[lm]
            print(f"{lm.name}: x={x:.1f}px y={y:.1f}px z={z:.1f}")
        out = draw_landmarks(sample_img.copy(), landmarks)
        plt.figure(figsize=[10, 10]); plt.title("Output"); plt.axis("off"); plt.imshow(out[:, :, ::-1]); plt.show()

### 3D view of the world landmarks

In [ ]:
def plot_world_landmarks(world):
    fig = plt.figure(figsize=[8, 8])
    ax = fig.add_subplot(projection="3d")
    xs, ys, zs = [p.x for p in world], [p.y for p in world], [p.z for p in world]
    # MediaPipe: y points down and z toward the camera. Plot with z as depth and -y as height.
    ax.scatter(zs, xs, [-y for y in ys], c="r", s=12)
    for a, b in POSE_CONNECTIONS:
        ax.plot([zs[a], zs[b]], [xs[a], xs[b]], [-ys[a], -ys[b]], c="k", lw=1)
    ax.set_xlabel("z (m)"); ax.set_ylabel("x (m)"); ax.set_zlabel("height (m)")
    ax.view_init(elev=10, azim=10)
    plt.show()

if IMAGES and world is not None:
    plot_world_landmarks(world)

## Angle between three landmarks

The angle is measured at the middle landmark, counter-clockwise in image coordinates, and lies in
[0, 360). Swapping the outer points gives `360 - angle`.

In [ ]:
print(f"The calculated angle is {calculate_angle((558, 326, 0), (642, 333, 0), (718, 321, 0))}")

## Classification rules

| Pose | Elbows | Shoulders | Knees |
|---|---|---|---|
| Warrior II | both 165-195 | both 80-110 | one 165-195, the other 90-120 |
| T Pose | both 165-195 | both 80-110 | both 160-195 |
| Tree Pose | any | any | one 165-195, the other 315-335 (left) or 25-45 (right) |

The rules depend on the camera angle, so they work best when the person faces the camera
directly. They live in `asana_abhyas/classifier.py`.

In [ ]:
def classify_image(path):
    image = cv2.imread(str(path))
    landmarks, _ = detector.detect(image)
    if landmarks is None:
        return image, "No person detected", None
    label, angles = classify_pose(landmarks)
    draw_landmarks(image, landmarks)
    draw_label(image, label)
    return image, label, angles

for path in IMAGES:
    out, label, angles = classify_image(path)
    print(f"{path.name}: {label}")
    if angles:
        print("   " + ", ".join(f"{k}={v:.0f}" for k, v in vars(angles).items()))
    plt.figure(figsize=[8, 8]); plt.title(f"{path.name}: {label}"); plt.axis("off"); plt.imshow(out[:, :, ::-1]); plt.show()

## Real-time classification on a webcam

This needs a camera and a desktop session, so it is off by default. Set `RUN_WEBCAM = True` to run
it, and press ESC or `q` to stop. The same loop is available from the terminal:
`python -m asana_abhyas --camera 0`.

In [ ]:
RUN_WEBCAM = False
CAMERA_INDEX = 0

if RUN_WEBCAM:
    from asana_abhyas.__main__ import main
    main(["--camera", str(CAMERA_INDEX)])

## Extending the rules

Other poses can be added in the same way. For example, the cobra pose (Bhujangasana) could be
recognised from both knees at about 180 degrees and both hips at about 105 degrees (facing right)
or 240 degrees (facing left). Widening the angle bands makes the poses easier to hit but also
produces more false positives.

In [ ]:
detector.close()